# 19 — Credit Default Swaps

- **Midpoint CDS engine** (instrument-based)
- **Analytics CDS engine** (array-based)
- **ISDA standard engine** (with accrual on default)
- Fair spread, NPV, risky annuity
- QuantLib comparison
- Credit term structure bootstrapping

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, load_quantlib, compare_table, timed_ms
jax = setup_backend(BACKEND)
import jax.numpy as jnp
import numpy as np
QL = load_quantlib()

In [ ]:
from ql_jax.instruments.cds import make_cds
from ql_jax.engines.credit.midpoint import midpoint_cds_npv, cds_fair_spread, hazard_rate_from_spread
from ql_jax.engines.credit.analytics import (
    cds_npv as analytics_cds_npv,
    cds_fair_spread as analytics_fair_spread)
from ql_jax.engines.credit.isda import isda_cds_npv, isda_cds_fair_spread

NOTIONAL = 1_000_000.0
SPREAD = 0.015         # 150 bp
MATURITY = 5.0
RECOVERY = 0.40
HAZARD_RATE = 0.02
DISC_RATE = 0.03

---
## 1. CDS Instrument

In [ ]:
cds = make_cds(NOTIONAL, SPREAD, MATURITY, RECOVERY, frequency=0.25)

print(f"Notional  : {cds.notional:,.0f}")
print(f"Spread    : {cds.spread*10000:.0f} bp")
print(f"Maturity  : {cds.maturity} years")
print(f"Recovery  : {cds.recovery_rate:.0%}")
print(f"Payments  : {cds.n_periods} (quarterly)")

---
## 2. Midpoint Engine

In [ ]:
disc_fn = lambda t: jnp.exp(-DISC_RATE * t)
surv_fn = lambda t: jnp.exp(-HAZARD_RATE * t)

npv_mid = float(midpoint_cds_npv(cds, disc_fn, surv_fn))
fair_mid = float(cds_fair_spread(cds, disc_fn, surv_fn))

print(f"Midpoint NPV         : {npv_mid:12.2f}")
print(f"Midpoint fair spread : {fair_mid*10000:8.2f} bp")

---
## 3. ISDA Engine

In [ ]:
payment_dates = jnp.array(cds.payment_dates)
day_fracs = jnp.full_like(payment_dates, 0.25)

npv_isda = float(isda_cds_npv(
    SPREAD, RECOVERY, payment_dates, day_fracs,
    disc_fn, surv_fn, NOTIONAL, is_buyer=True, accrual_on_default=True))

fair_isda = float(isda_cds_fair_spread(
    RECOVERY, payment_dates, day_fracs, disc_fn, surv_fn, accrual_on_default=True))

print(f"ISDA NPV             : {npv_isda:12.2f}")
print(f"ISDA fair spread     : {fair_isda*10000:8.2f} bp")

---
## 4. QuantLib Comparison

In [ ]:
today = QL.Date(15, 6, 2024)
QL.Settings.instance().evaluationDate = today

dc = QL.Actual365Fixed()
r_ts = QL.YieldTermStructureHandle(QL.FlatForward(today, DISC_RATE, dc))
hz_ts = QL.DefaultProbabilityTermStructureHandle(
    QL.FlatHazardRate(today, HAZARD_RATE, dc))

ql_cds = QL.CreditDefaultSwap(
    QL.Protection.Buyer,
    NOTIONAL, SPREAD,
    QL.Schedule(today, today + QL.Period(int(MATURITY), QL.Years),
                QL.Period(QL.Quarterly), QL.TARGET(),
                QL.Following, QL.Following,
                QL.DateGeneration.Forward, False),
    QL.Following, dc)

engine = QL.MidPointCdsEngine(hz_ts, RECOVERY, r_ts)
ql_cds.setPricingEngine(engine)

ql_npv = ql_cds.NPV()
ql_fair = ql_cds.fairSpread()

print(f"\n{'Engine':<20s} {'NPV':>12s} {'Fair (bp)':>10s}")
print("-" * 45)
print(f"{'QL midpoint':<20s} {ql_npv:>12.2f} {ql_fair*10000:>10.2f}")
print(f"{'ql-jax midpoint':<20s} {npv_mid:>12.2f} {fair_mid*10000:>10.2f}")
print(f"{'ql-jax ISDA':<20s} {npv_isda:>12.2f} {fair_isda*10000:>10.2f}")

---
## 5. Credit Curve Bootstrapping

In [ ]:
from ql_jax.termstructures.credit.helpers import CdsHelper, bootstrap_default_curve

market_spreads = [(1.0, 0.008), (2.0, 0.010), (3.0, 0.012),
                  (5.0, 0.015), (7.0, 0.018), (10.0, 0.020)]

helpers = [CdsHelper(market_spread=s, maturity=m, recovery_rate=RECOVERY)
           for m, s in market_spreads]

result = bootstrap_default_curve(helpers, disc_fn)

print(f"{'Maturity':<10s} {'Spread (bp)':<12s} {'Hazard Rate':<12s} {'Survival':<12s}")
print("-" * 50)
for (m, s), h, q in zip(market_spreads, result['hazard_rates'], result['survival_probabilities']):
    print(f"{m:<10.0f} {s*10000:<12.0f} {float(h):<12.6f} {float(q):<12.6f}")

---
## 6. CDS Spread vs Maturity

In [ ]:
import matplotlib.pyplot as plt

mats = np.arange(0.5, 10.5, 0.5)
fair_spreads = []
for m in mats:
    c = make_cds(NOTIONAL, 0.01, float(m), RECOVERY)
    fs = float(cds_fair_spread(c, disc_fn, surv_fn))
    fair_spreads.append(fs * 10000)

plt.figure(figsize=(8, 5))
plt.plot(mats, fair_spreads, 'bo-', markersize=5)
plt.xlabel('Maturity (years)')
plt.ylabel('Fair Spread (bp)')
plt.title(f'CDS Fair Spread Term Structure (h={HAZARD_RATE}, r={DISC_RATE})')
plt.grid(True, alpha=0.3)
plt.show()

---
## 7. Exercises

1. **Recovery sensitivity**: Plot fair spread vs recovery rate from 10% to 80%.
2. **Upfront vs running**: Compute the upfront amount for a 500bp coupon CDS using ISDA engine.
3. **Hazard from spread**: Use `hazard_rate_from_spread` to verify the flat hazard approximation: $h \approx s / (1-R)$.

---
*End of Notebook 19*